In [2]:
import pprint
import radiate as rd
import numpy as np
import polars as pl

rd.random.seed(123)

In [ ]:
def compute(x: float) -> float:
    return 4.0 * x**3 - 3.0 * x**2 + x


inputs = []
answers = []

x = -1.0
for _ in range(20):
    x += 0.1
    inputs.append([x])
    answers.append([compute(x)])

X = np.array(inputs, dtype=np.float32)  # (N, 1)
Y = np.array(answers, dtype=np.float32)  # (N, 1)

Xb = np.concatenate([X, np.ones((X.shape[0], 1), dtype=np.float32)], axis=1)


def fit(weights: list[np.ndarray]) -> float:
    W1 = weights[0].reshape((8, 2))
    W2 = weights[1].reshape((8, 8))
    W3 = weights[2].reshape((1, 8))

    h1 = Xb @ W1.T
    h1 = np.maximum(0, h1)

    h2 = h1 @ W2
    h2 = np.tanh(h2)

    yhat = h2 @ W3.T

    return float(np.mean((yhat - Y) ** 2, dtype=np.float32))

In [4]:
collector = rd.MetricCollector()
engine = (
    rd.Engine.float(
        shape=[16, 64, 8],
        init_range=(-1.0, 1.0),
        bounds=(-3.0, 3.0),
        use_numpy=True,
        dtype=rd.Float32,
    )
    .fitness(fit)
    .subscribe(collector)
    .minimizing()
    .select(rd.Select.boltzmann(temp=4.0))
    .alters(rd.Cross.blend(0.7, 0.4), rd.Mutate.gaussian(0.3))
    .limit(rd.Limit.score(0.01), rd.Limit.generations(500))
)

result = engine.run(log=True)

2026-04-09T15:02:14.524199Z  INFO Epoch 1    | Score:   1.1841 | Time: 4.88ms
2026-04-09T15:02:14.525435Z  INFO Epoch 2    | Score:   0.9343 | Time: 5.99ms
2026-04-09T15:02:14.526718Z  INFO Epoch 3    | Score:   0.8793 | Time: 6.95ms
2026-04-09T15:02:14.527626Z  INFO Epoch 4    | Score:   0.8793 | Time: 7.81ms
2026-04-09T15:02:14.528558Z  INFO Epoch 5    | Score:   0.8697 | Time: 8.70ms
2026-04-09T15:02:14.529446Z  INFO Epoch 6    | Score:   0.7182 | Time: 9.55ms
2026-04-09T15:02:14.530338Z  INFO Epoch 7    | Score:   0.7182 | Time: 10.41ms
2026-04-09T15:02:14.531384Z  INFO Epoch 8    | Score:   0.5674 | Time: 11.36ms
2026-04-09T15:02:14.532450Z  INFO Epoch 9    | Score:   0.5674 | Time: 12.37ms
2026-04-09T15:02:14.533367Z  INFO Epoch 10   | Score:   0.5674 | Time: 13.24ms
2026-04-09T15:02:14.534230Z  INFO Epoch 11   | Score:   0.5085 | Time: 14.08ms
2026-04-09T15:02:14.535100Z  INFO Epoch 12   | Score:   0.3981 | Time: 14.92ms
2026-04-09T15:02:14.535963Z  INFO Epoch 13   | Score:   0.

In [5]:
metrics = result.metrics()
df = metrics.to_polars()
df

name,last,sum,mean,stddev,var,skew,min,max,count,time_sum,time_mean,time_stddev,time_min,time_max,time_var,version,update_count,tags
str,f64,f64,f64,f64,f64,f64,f64,f64,i64,duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],i64,i64,list[str]
"""lineage_events""",80.0,40000.0,80.0,0.0,0.0,0.0,80.0,80.0,500,null,null,null,null,null,null,500,1,"[""age"", ""statistic"", ""lineage""]"
"""blend_crossover_rate""",0.7,350.0,0.7,0.0,0.0,0.0,0.7,0.7,500,null,null,null,null,null,null,500,1,"[""alterer"", ""crossover"", … ""rate""]"
"""unique_scores""",97.0,47885.0,95.769997,1.985191,3.940982,-0.463828,90.0,100.0,500,null,null,null,null,null,null,500,1,"[""derived"", ""statistic"", ""score""]"
"""filter_step""",0.0,null,null,null,null,null,null,null,500,3258µs,6µs,3µs,5µs,69µs,0µs,500,1,"[""time"", ""step""]"
"""alter_cross_family""",43.0,27071.0,44.671616,20.194004,407.797791,-1.554685,1.0,64.0,606,null,null,null,null,null,null,500,1,"[""alterer"", ""statistic"", … ""lineage""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""lineage_family_pair_unique""",1.0,606.0,1.212,2.233615,4.989034,18.03718,1.0,36.0,500,null,null,null,null,null,null,500,1,"[""age"", ""statistic"", ""lineage""]"
"""lineage_family_pair""",0.0,7.269003,0.014538,0.102937,0.010596,0.0,0.0,0.978788,500,null,null,null,null,null,null,500,1,"[""age"", ""statistic"", ""lineage""]"
"""diversity_ratio""",0.97,478.850006,0.9577,0.019852,0.000394,0.0,0.9,1.0,500,null,null,null,null,null,null,500,1,"[""derived"", ""statistic""]"


In [6]:
print(result.metrics().dashboard())
# pprint.pprint(metrics["carryover_rate"].tags())

for metric in metrics.values_by_tag(rd.Tag.DERIVED):
    print(metric)

print()
pprint.pprint(metrics["carryover_rate"].to_dict())

[34 metrics, 16021 updates]
----- Metrics -----
  carryover: 0.164  diversity: 0.958  unique_members: 97  unique_scores: 97  improvements: 17  iter_time(mean): 881.016µs

== Statistics ==
Name                     | Type   | Mean       | Min        | Max        | N      | Total        | StdDev     | Skew       | Kurt       | Entr      
-------------------------------------------------------------------------------------------------------------------------------------------------
age                      | value  | 0.426      | 0.000      | 20.000     | 50000  | -            | 1.390      | 19.663     | 263.791    | -         
alter_cross_family       | value  | 44.672     | 1.000      | 64.000     | 606    | -            | 20.194     | -1.555     | 3.655      | -         
alter_parent_reuse       | value  | 2.637      | 1.000      | 58.000     | 25438  | -            | 4.827      | 5.780      | 41.011     | -         
alter_within_family      | value  | 105.292    | 1.000      | 144.000 

In [ ]:
df = collector.to_polars(lazy=False)

df = (
    df.filter(
        (pl.col("update_count") > 1) & (~pl.col("tags").list.contains("distribution"))
    )
    .select("name", "version", "update_count", "tags")
    .group_by("name")
    .agg(
        pl.col("update_count").sum().alias("update_count"),
    )
    .sort("update_count", descending=False)
)

df

name,update_count
str,i64
"""evaluation_count""",2
"""blend_crossover""",998
"""gaussian_mutator""",998
"""roulette_selector""",1000
"""boltzmann_selector""",1000
"""evaluate_step""",1000


In [ ]:
df = collector.to_polars(lazy=False)


In [ ]:
df.filter(pl.col("name") == "gaussian_mutator")

name,last,sum,mean,stddev,var,skew,min,max,count,time_sum,time_mean,time_stddev,time_min,time_max,time_var,version,update_count,tags
str,f64,f64,f64,f64,f64,f64,f64,f64,i64,duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],i64,i64,list[str]
"""gaussian_mutator""",2148.0,2148.0,2148.0,0.0,0.0,NaN,2148.0,2148.0,1,80µs,80µs,0µs,80µs,80µs,0µs,1,0,"[""alterer"", ""mutator"", … ""time""]"
"""gaussian_mutator""",2098.0,4246.0,2123.0,35.355339,1250.0,NaN,2098.0,2148.0,2,157µs,78µs,1µs,77µs,80µs,0µs,2,2,"[""alterer"", ""mutator"", … ""time""]"
"""gaussian_mutator""",2082.0,6328.0,2109.333252,34.428669,1185.333374,2.431087,2082.0,2148.0,3,228µs,76µs,4µs,70µs,80µs,0µs,3,2,"[""alterer"", ""mutator"", … ""time""]"
"""gaussian_mutator""",2103.0,8431.0,2107.75,28.288687,800.249756,2.140824,2082.0,2148.0,4,299µs,74µs,4µs,70µs,80µs,0µs,4,2,"[""alterer"", ""mutator"", … ""time""]"
"""gaussian_mutator""",2059.0,10490.0,2098.0,32.794811,1075.499756,1.020588,2059.0,2148.0,5,371µs,74µs,4µs,70µs,80µs,0µs,5,2,"[""alterer"", ""mutator"", … ""time""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""gaussian_mutator""",2075.0,1.046253e6,2109.381104,42.267235,1786.519165,-0.021626,1974.0,2248.0,496,35716µs,72µs,4µs,66µs,113µs,0µs,496,2,"[""alterer"", ""mutator"", … ""time""]"
"""gaussian_mutator""",2165.0,1.048418e6,2109.49292,42.298244,1789.141479,-0.024914,1974.0,2248.0,497,35788µs,72µs,4µs,66µs,113µs,0µs,497,2,"[""alterer"", ""mutator"", … ""time""]"
"""gaussian_mutator""",2068.0,1.050486e6,2109.409668,42.296558,1788.998901,-0.020825,1974.0,2248.0,498,35862µs,72µs,4µs,66µs,113µs,0µs,498,2,"[""alterer"", ""mutator"", … ""time""]"
